In [98]:
import gensim
from gensim.models import Word2Vec, KeyedVectors

In [99]:
import pandas as pd

messages = pd.read_csv('data/SMSSpamCollection', sep='\t', names=['label', 'message'])

messages.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [100]:
len(messages)

5572

In [101]:
import nltk
nltk.download('wordnet')
nltk.download('punkt_tab')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [102]:
from nltk.stem import WordNetLemmatizer

In [103]:
lemmatizer = WordNetLemmatizer()

In [104]:
import re

corpus = []

for i in range(len(messages)):
    message = re.sub("[^a-zA-Z]", " ", messages["message"][i])
    message = message.lower()
    message = message.split()

    message = [
        lemmatizer.lemmatize(word) for word in message
    ]

    message = " ".join(message)

    corpus.append(message)

In [105]:
from nltk import sent_tokenize
from gensim.utils import simple_preprocess

In [106]:
words = []
for sent in corpus:
  sent_token = sent_tokenize(sent)
  for sent_item in sent_token:
    words.append(simple_preprocess(sent_item))

In [107]:
# now words is a 2d array, with single arrays as (all words in single sentence)
words[0]

['go',
 'until',
 'jurong',
 'point',
 'crazy',
 'available',
 'only',
 'in',
 'bugis',
 'great',
 'world',
 'la',
 'buffet',
 'cine',
 'there',
 'got',
 'amore',
 'wat']

In [108]:
model = Word2Vec(words)

In [109]:
# get all the vocabulary
# model.wv.index_to_key

In [110]:
model.corpus_count

5569

In [111]:
model.epochs

5

In [112]:
model.wv.similar_by_word("good")

[('my', 0.9988624453544617),
 ('day', 0.9988046288490295),
 ('hope', 0.9987305402755737),
 ('great', 0.9987278580665588),
 ('and', 0.9987160563468933),
 ('babe', 0.9986563920974731),
 ('all', 0.9986497163772583),
 ('of', 0.998620867729187),
 ('well', 0.9986134171485901),
 ('about', 0.9985620975494385)]

In [113]:
model.wv['good'].shape

(100,)

In [114]:
import numpy as np


def avg_word2vec(doc):
    vectors = [model.wv[word] for word in doc if word in model.wv.index_to_key]
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

In [115]:
from tqdm import tqdm

In [116]:
X = []
for i in tqdm(range(len(words))):
  X.append(avg_word2vec(words[i]))

100%|██████████| 5569/5569 [00:00<00:00, 8582.10it/s]


In [117]:
X = np.array(X)

In [118]:
# X[0] is a single sentence, now a single sentence have 100 dimensions, before every word had 100 dimensions, like if a sentence had 10 words then that sentence had 1000 dimensions (10 x 100)
X[0].shape

(100,)

In [119]:
# Encoding dependent variable: ham -> 0, spam -> 1
y = messages[list(map(lambda x: len(x)>0 ,corpus))]
y=pd.get_dummies(y['label'])
y=y.iloc[:,0].values

y.shape

(5569,)

In [120]:
df = pd.DataFrame(X)

rows = [pd.DataFrame(X[i].reshape(1, -1)) for i in range(len(X))]
df = pd.concat(rows, ignore_index=True)

In [121]:
df.shape

(5569, 100)

In [122]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df, y, test_size=0.20, random_state=0
)

## Modeling

In [123]:
from sklearn.ensemble import RandomForestClassifier

In [124]:
classifier = RandomForestClassifier()

In [125]:
classifier.fit(X_train, y_train)

RandomForestClassifier()

In [126]:
y_pred = classifier.predict(X_test)

In [127]:
from sklearn.metrics import accuracy_score, classification_report

print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

0.9649910233393177
              precision    recall  f1-score   support

       False       0.89      0.86      0.87       154
        True       0.98      0.98      0.98       960

    accuracy                           0.96      1114
   macro avg       0.93      0.92      0.93      1114
weighted avg       0.96      0.96      0.96      1114

